In [40]:
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.datasets import mnist
from tensorflow.keras.layers import Input, Dense, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras import backend as K
from tensorflow.keras import losses
from scipy.stats import norm

In [41]:
import tensorflow
print(tensorflow.__version__)

2.19.0


In [ ]:
# Load data
(x_tr, y_tr), (x_te, y_te) = mnist.load_data()

In [ ]:
# Normalize and Reshape images (flatten)
x_tr, x_te = x_tr.astype('float32')/255., x_te.astype('float32')/255. # Normalize pixel values to [0, 1]
x_tr_flat, x_te_flat = x_tr.reshape(x_tr.shape[0], -1), x_te.reshape(x_te.shape[0], -1) # Flatten images to 1D vectors

In [ ]:
print(x_tr.shape, x_te.shape)
print(x_tr_flat.shape, x_te_flat.shape)

In [ ]:
# Neural Network Parameters
batch_size, n_epoch = 100, 50
n_hidden, z_dim = 256, 2 # start with 256 to 2 vector embeddings

In [ ]:
# Example of a training image
plt.imshow(x_tr[1]);

In [ ]:
# Sampling Function
def sampling(args):
    mu, log_var = args # unpack mean and log-variance (output of encoder)
    eps = K.random_normal(shape=(batch_size, z_dim), mean=0, stddev=1.0) # Sample epsilon from N(0, I) - random noise
    return mu + K.exp(log_var) * eps # returns a sample from latent distribution z = mu + sigma * epsilon

In [ ]:
# Encoder - from 784->256->128->2
inputs_flat = Input(shape=(x_tr_flat.shape[1:])) # Input layer with 784 features (flattened image)
x_flat = Dense(n_hidden, activation='relu')(inputs_flat) # First hidden layer with 256 units
x_flat = Dense(n_hidden, activation='relu')(x_flat) # Second hidden layer with 256 units

# The network outputs two vectors: mu_flat (mean) and log_var_flat (log-variance) for the latent space.
mu_flat = Dense(z_dim)(x_flat) # Mean of the latent space
log_var_flat = Dense(z_dim)(x_flat) # Log-variance of the latent space
z_flat = Lambda(sampling, output_shape=(z_dim, ))([mu_flat, log_var_flat]) # Create a Sampling layer to generate latent vector

# Lambda(sampling, output_shape=(z_dim, )) creates a custom layer that applies your sampling function.
# It takes [mu_flat, log_var_flat] as input, which are the mean and log-variance vectors from the encoder.
# The sampling function uses these to generate a random latent vector z_flat using the reparameterization trick.
# z_flat will have shape (batch_size, z_dim) and represents a sample from the learned latent space distribution for each input.

# In a VAE, we want to sample a latent vector z from a distribution defined by the encoder's outputs (mu_flat and log_var_flat).
# Direct sampling would break the computational graph, making backpropagation (and thus training) impossible.
# The reparameterization trick expresses the sampling as a deterministic function of mu, log_var, and random noise, allowing gradients to flow through the network.

In [ ]:
# Decoder - from 2->128->256->784
latent_inputs = Input(shape=(z_dim,)) # Input layer for latent space (2 dimensions)
z_decoder1 = Dense(n_hidden//2, activation='relu')
z_decoder2 = Dense(n_hidden, activation='relu')
y_decoder = Dense(x_tr_flat.shape[1], activation='sigmoid') # Output layer with 784 features (flattened image)

z_decoded = z_decoder1(latent_inputs) # First hidden layer with 128 units
z_decoded = z_decoder2(z_decoded) # Second hidden layer with 256 units
y_decoded = y_decoder(z_decoded) # Output layer with 784 units (flattened image)

# Define the VAE model
decoder_flat = Model(latent_inputs, y_decoded, name='decoder_conv')
outputs_flat = decoder_flat(z_flat)

In [ ]:
# variational autoencoder (VAE) - to reconstruction input
reconstruction_loss = losses.binary_crossentropy(inputs_flat, outputs_flat) * x_tr_flat.shape[1]
kl_loss = 0.5 * K.sum(K.square(mu_flat) + K.exp(log_var_flat) - log_var_flat - 1, axis=-1)
vae_flat_loss = reconstruction_loss + kl_loss

# Define the VAE model
vae_flat = Model(inputs_flat, outputs_flat)
vae_flat.add_loss(vae_flat_loss)  # Add the loss to the model
vae_flat.compile(optimizer='adam')

In [ ]:
# train
vae_flat.fit(
    x_tr_flat,
    shuffle=True,
    epochs=n_epoch,
    batch_size=batch_size,
    validation_data=(x_te_flat, None),
    verbose=1
)